# 📘 Notebook 02: Joins and Aggregation

### Course: *From One Table to a Cluster: Working with Data at Scale*
**Instructor:** Ioannis Tsioulis

*Module A: Data at Rest · Notebook 2 of 4 in this module · (2 of 12 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- **Join** tables on their keys, and write a join **by hand** in two different ways
- Explain why one way of joining is thousands of times faster than the other
- Choose between an **inner** join and a **left** join
- Summarise many rows with **aggregate functions** and `GROUP BY`
- Tell `WHERE` from `HAVING` by knowing the order in which a query is evaluated
- Use **window functions** for rankings and running totals
- Recognise the commonest way a join silently produces **wrong numbers**

> **Prerequisites:** Notebook 01. Dictionaries. The idea that a dictionary finds a key in constant time is explained in *From Zero to Data Structures & Algorithms in Python*, Notebook 17 (hash tables).
>
> 🔭 **Why this notebook matters:** in Notebook 01 we split the data into four tables so that every fact is stored once. Real questions do not respect that split. *"How much did we earn from science books in Patras last March?"* touches all four tables at once. Joining puts the pieces back together, and aggregation turns thousands of rows into the one number somebody asked for. These two operations are most of what anyone ever does with a database.

> 📦 **Libraries used in this notebook.** None that need installing: only **SQLite**, built into Python and introduced in Notebook 01.

In [ ]:
import random
import datetime
import sqlite3
import time

The cell below rebuilds the bookshop database of Notebook 01. It is the same in every notebook of this module, so you can run it and move on.

In [ ]:
def make_shop(customers=1000, products=200, orders=10000, seed=0):
    rng = random.Random(seed)
    cities = ["Athens", "Thessaloniki", "Patras", "Heraklion", "Larissa", "Volos", "Ioannina", "Chania"]
    categories = ["Fiction", "Science", "History", "Children", "Cooking", "Travel", "Art", "Computing"]
    customer_rows = [(i, f"Customer {i}", rng.choice(cities), 1950 + rng.randint(0, 55)) for i in range(1, customers + 1)]
    product_rows = [(i, f"Book {i}", rng.choice(categories), round(rng.uniform(5, 60), 2)) for i in range(1, products + 1)]
    order_rows, item_rows = [], []
    first_day = datetime.date(2023, 1, 1)
    for order_id in range(1, orders + 1):
        day = first_day + datetime.timedelta(days=rng.randint(0, 729))
        order_rows.append((order_id, rng.randint(1, customers), day.isoformat()))
        for line in range(rng.randint(1, 4)):
            item_rows.append((len(item_rows) + 1, order_id, rng.randint(1, products), rng.randint(1, 3)))
    return customer_rows, product_rows, order_rows, item_rows

def build_database(customer_rows, product_rows, order_rows, item_rows):
    db = sqlite3.connect(":memory:")
    db.executescript("""
        CREATE TABLE customers   (customer_id INTEGER PRIMARY KEY, name TEXT NOT NULL, city TEXT, birth_year INTEGER);
        CREATE TABLE products    (product_id INTEGER PRIMARY KEY, title TEXT NOT NULL, category TEXT, price REAL);
        CREATE TABLE orders      (order_id INTEGER PRIMARY KEY, customer_id INTEGER NOT NULL REFERENCES customers(customer_id), order_date TEXT);
        CREATE TABLE order_items (item_id INTEGER PRIMARY KEY, order_id INTEGER NOT NULL REFERENCES orders(order_id),
                                  product_id INTEGER NOT NULL REFERENCES products(product_id), quantity INTEGER);
    """)
    db.executemany("INSERT INTO customers VALUES (?, ?, ?, ?)", customer_rows)
    db.executemany("INSERT INTO products VALUES (?, ?, ?, ?)", product_rows)
    db.executemany("INSERT INTO orders VALUES (?, ?, ?)", order_rows)
    db.executemany("INSERT INTO order_items VALUES (?, ?, ?, ?)", item_rows)
    db.commit()
    return db

customer_rows, product_rows, order_rows, item_rows = make_shop()
db = build_database(customer_rows, product_rows, order_rows, item_rows)

def show(sql, parameters=(), limit=8):
    cursor = db.execute(sql, parameters)
    columns = [description[0] for description in cursor.description]
    rows = cursor.fetchall()
    print(" | ".join(f"{column:<14}" for column in columns))
    print("-" * (17 * len(columns) - 3))
    for row in rows[:limit]:
        print(" | ".join(f"{str(value):<14}" for value in row))
    if len(rows) > limit:
        print(f"... {len(rows):,} rows in total")
    print()

print(len(customer_rows), "customers,", len(product_rows), "products,", len(order_rows), "orders,", len(item_rows), "order items")

## 1. Putting tables back together

### Intuition first
The `orders` table says that order 1 was placed by customer number so-and-so. To print the customer's **name** next to the order, we must look that number up in the `customers` table. A **join** does this lookup for every row at once: it lays two tables side by side, matching each row of one with the rows of the other that share a key.

### A join by hand
The most direct way: for every order, go through all the customers until the matching one is found.

In [ ]:
def nested_loop_join(orders, customers):
    result = []
    for order_id, customer_id, order_date in orders:
        for other_id, name, city, birth_year in customers:
            if customer_id == other_id:                  # the join condition
                result.append((order_id, order_date, name, city))
    return result

started = time.perf_counter()
joined = nested_loop_join(order_rows, customer_rows)
nested_time = time.perf_counter() - started

print(joined[:3])
print(f"{len(joined):,} rows in {nested_time:.2f} seconds")
print(f"Comparisons made: {len(order_rows) * len(customer_rows):,}")

It is correct, and it does ten million comparisons to match ten thousand orders. The work is the **product** of the two table sizes. Double both tables and it takes four times as long.

### A better join by hand
We are looking each customer up by their id, again and again. Looking things up by key is what a **dictionary** is for. So first build a dictionary of the customers, once, and then go through the orders a single time.

In [ ]:
def hash_join(orders, customers):
    by_id = {}
    for customer_id, name, city, birth_year in customers:        # build: one pass over the smaller table
        by_id[customer_id] = (name, city)

    result = []
    for order_id, customer_id, order_date in orders:             # probe: one pass over the larger table
        if customer_id in by_id:
            name, city = by_id[customer_id]
            result.append((order_id, order_date, name, city))
    return result

started = time.perf_counter()
joined_again = hash_join(order_rows, customer_rows)
hash_time = time.perf_counter() - started

print(f"{len(joined_again):,} rows in {hash_time:.4f} seconds")
print("Same result:", sorted(joined) == sorted(joined_again))
print(f"About {nested_time / hash_time:,.0f} times faster")

The same answer, hundreds of times faster. The work is now the **sum** of the two table sizes and not their product. This algorithm is called a **hash join**, and it is one of the methods a real database uses. The other well-known ones are the nested loop you wrote first, which is fine when one table is tiny or an index is available (Notebook 03), and the *sort-merge join*, which sorts both tables by the key and walks down them together.

### The same join in SQL

In [ ]:
show("""
    SELECT orders.order_id, orders.order_date, customers.name, customers.city
    FROM orders
    JOIN customers ON orders.customer_id = customers.customer_id
    ORDER BY orders.order_id
""", limit=5)

### Formal definition
An **inner join** of tables $A$ and $B$ on a condition returns one row for every pair (a row of $A$, a row of $B$) that satisfies the condition. `JOIN ... ON` states the condition. You do **not** say which algorithm to use. The database chooses, which is the declarative idea of Notebook 01 at work.

Several joins can be chained. To answer *"which books did customer 7 buy?"* we walk across all four tables. Short names for the tables (`o`, `i`, `p`) keep the query readable.

In [ ]:
show("""
    SELECT o.order_date, p.title, p.category, i.quantity, p.price
    FROM customers AS c
    JOIN orders AS o       ON o.customer_id = c.customer_id
    JOIN order_items AS i  ON i.order_id = o.order_id
    JOIN products AS p     ON p.product_id = i.product_id
    WHERE c.customer_id = 7
    ORDER BY o.order_date
""", limit=6)

## 2. Rows without a partner: the left join

An inner join returns only the pairs that match. A row with no partner simply disappears. Often that is exactly the row you were looking for.

Let us add a new book that nobody has bought yet, and a customer who has not ordered anything.

In [ ]:
new_product = (201, "Book 201 (new)", "Fiction", 19.90)
new_customer = (1001, "Customer 1001 (new)", "Volos", 2001)

db.execute("INSERT INTO products VALUES (?, ?, ?, ?)", new_product)
db.execute("INSERT INTO customers VALUES (?, ?, ?, ?)", new_customer)
db.commit()
product_rows.append(new_product)          # keep the Python lists in step with the database
customer_rows.append(new_customer)

show("""
    SELECT p.title, i.item_id
    FROM products AS p
    JOIN order_items AS i ON i.product_id = p.product_id
    WHERE p.product_id = 201
""")

No rows. The new book has no order items, so the inner join has nothing to pair it with.

### Formal definition
A **left join** returns every row of the left-hand table. Where a row has no partner on the right, the right-hand columns are filled with `NULL`.

In [ ]:
show("""
    SELECT p.title, i.item_id
    FROM products AS p
    LEFT JOIN order_items AS i ON i.product_id = p.product_id
    WHERE p.product_id = 201
""")

The book is there, with `None` (Python's rendering of NULL) where the order item would be. This gives a standard recipe for *"find the rows that have no match"*: left join, then keep the rows where the right side is NULL.

In [ ]:
show("""
    SELECT c.customer_id, c.name
    FROM customers AS c
    LEFT JOIN orders AS o ON o.customer_id = c.customer_id
    WHERE o.order_id IS NULL
""")

## 3. Aggregation: many rows, one number

### Intuition first
Managers do not ask for twenty-five thousand order lines. They ask *how many*, *how much in total*, *what is the average*. An **aggregate function** reduces a whole column to a single value.

In [ ]:
show("""
    SELECT COUNT(*)            AS books,
           ROUND(AVG(price), 2) AS average_price,
           MIN(price)           AS cheapest,
           MAX(price)           AS dearest
    FROM products
""")

### One number per group
More often we want one number **for each** category, each city, each month. By hand, that is a dictionary with one entry per group.

In [ ]:
books_per_category = {}
for product_id, title, category, price in product_rows:
    books_per_category[category] = books_per_category.get(category, 0) + 1

for category in sorted(books_per_category):
    print(f"{category:<10} {books_per_category[category]}")

### Formal definition
`GROUP BY` splits the rows into groups that share a value, and each aggregate function is then computed **once per group**. The result has one row per group.

In [ ]:
show("""
    SELECT category, COUNT(*) AS books, ROUND(AVG(price), 2) AS average_price
    FROM products
    GROUP BY category
    ORDER BY category
""", limit=10)

Joins and grouping combine naturally. The revenue of an order line is its quantity times the price of the book, and the price is in another table.

In [ ]:
show("""
    SELECT p.category,
           SUM(i.quantity)                    AS copies_sold,
           ROUND(SUM(i.quantity * p.price))   AS revenue
    FROM order_items AS i
    JOIN products AS p ON p.product_id = i.product_id
    GROUP BY p.category
    ORDER BY revenue DESC
""", limit=10)

Twenty-five thousand rows in two tables, reduced to eight lines that someone can read.

### WHERE or HAVING?
Both filter, at different moments. `WHERE` discards **rows** before they are grouped. `HAVING` discards **groups** after the aggregates have been computed.

In [ ]:
show("""
    SELECT c.city, COUNT(*) AS orders_2024
    FROM orders AS o
    JOIN customers AS c ON c.customer_id = o.customer_id
    WHERE o.order_date >= '2024-01-01'          -- rows: only orders of 2024
    GROUP BY c.city
    HAVING COUNT(*) > 650                       -- groups: only the busiest cities
    ORDER BY orders_2024 DESC
""")

### The order in which a query is evaluated
A query is **written** starting with `SELECT`, but it is **evaluated** in a different order. Knowing that order explains most of the error messages you will ever get.

| Step | Clause | What happens |
|---|---|---|
| 1 | `FROM` and `JOIN` | the tables are combined |
| 2 | `WHERE` | rows are filtered |
| 3 | `GROUP BY` | the remaining rows are grouped |
| 4 | `HAVING` | groups are filtered |
| 5 | `SELECT` | the output columns are computed |
| 6 | `ORDER BY` | the result is sorted |
| 7 | `LIMIT` | the first rows are kept |

`WHERE` comes before the grouping, so it cannot mention `COUNT(*)`: the counts do not exist yet. `HAVING` comes after, so it can.

## 4. The join that lies

Here is a mistake that costs companies real money, because it produces a number that looks perfectly reasonable. How many orders did each city place?

In [ ]:
show("""
    SELECT c.city, COUNT(*) AS orders
    FROM customers AS c
    JOIN orders AS o       ON o.customer_id = c.customer_id
    JOIN order_items AS i  ON i.order_id = o.order_id
    GROUP BY c.city
    ORDER BY c.city
""", limit=3)

print("Total 'orders' counted this way:", db.execute("""
    SELECT COUNT(*) FROM customers c JOIN orders o ON o.customer_id = c.customer_id JOIN order_items i ON i.order_id = o.order_id
""").fetchone()[0])
print("Rows in the orders table:       ", db.execute("SELECT COUNT(*) FROM orders").fetchone()[0])

The query reports about two and a half times as many orders as exist. Nothing failed, and no warning was given.

The cause is the join with `order_items`. An order with three lines appears **three times** after that join, once for each line, and `COUNT(*)` counts rows. This is called **fan-out**: joining to a table with many rows per key multiplies the rows on the other side.

The repair is either to leave out the table that is not needed, or to count each order once with `COUNT(DISTINCT ...)`.

In [ ]:
show("""
    SELECT c.city, COUNT(DISTINCT o.order_id) AS orders, COUNT(*) AS order_lines
    FROM customers AS c
    JOIN orders AS o       ON o.customer_id = c.customer_id
    JOIN order_items AS i  ON i.order_id = o.order_id
    GROUP BY c.city
    ORDER BY c.city
""", limit=3)

> ⚠️ **Common pitfalls**
>
> - **Fan-out.** After every join, ask: *what does one row of this result represent?* If the answer has changed from "an order" to "an order line", every count and every sum has changed meaning with it.
> - Forgetting the `ON` condition. Without it you get every row of one table paired with every row of the other, a **cross join**: here, 10,000 × 1,000 rows.
> - Putting a column in `SELECT` that is neither grouped nor aggregated. Most databases refuse. SQLite quietly picks a value from some row of the group, which is worse.
> - Using an inner join when rows without a partner matter. Customers who never ordered vanish from an inner join, and with them the answer to *"who should we write to?"*.

## 5. Window functions: aggregates that keep the rows

### Intuition first
`GROUP BY` collapses each group into one row. Sometimes you want the summary **and** the individual rows: each book next to its rank within its category, or each month next to the running total so far. A **window function** computes an aggregate over a set of related rows, the *window*, and attaches the result to every row without collapsing anything.

### Formal definition
`function(...) OVER (PARTITION BY ... ORDER BY ...)`

- `PARTITION BY` divides the rows into groups, like `GROUP BY`, but the rows are kept.
- `ORDER BY` inside `OVER` puts the rows of each group in order, which is what a ranking or a running total needs.

First, the three best-selling books of each category. The inner query computes each book's revenue and its rank within its category. The outer query keeps the top three.

In [ ]:
show("""
    SELECT category, title, revenue, position
    FROM (
        SELECT p.category, p.title,
               ROUND(SUM(i.quantity * p.price)) AS revenue,
               RANK() OVER (PARTITION BY p.category ORDER BY SUM(i.quantity * p.price) DESC) AS position
        FROM order_items AS i
        JOIN products AS p ON p.product_id = i.product_id
        GROUP BY p.product_id
    )
    WHERE position <= 3
    ORDER BY category, position
""", limit=9)

Second, monthly revenue with a running total. `SUM(...) OVER (ORDER BY month)` adds up everything from the first month to the current row.

In [ ]:
show("""
    SELECT month, revenue, SUM(revenue) OVER (ORDER BY month) AS running_total
    FROM (
        SELECT SUBSTR(o.order_date, 1, 7) AS month, ROUND(SUM(i.quantity * p.price)) AS revenue
        FROM orders AS o
        JOIN order_items AS i ON i.order_id = o.order_id
        JOIN products AS p    ON p.product_id = i.product_id
        GROUP BY month
    )
    ORDER BY month
""", limit=6)

Without window functions, both of these would need the table to be joined with itself in awkward ways. With them, questions about *rank*, *previous row* and *total so far* become one line each. They are available in every modern database, and in Spark (Notebook 10).

## 6. Where we stand

| The problem | The answer |
|---|---|
| The answer to a question is spread over several tables | joins on keys |
| Matching rows one against all is too slow | a hash join: build a dictionary, then probe it |
| Rows without a partner disappear | a left join |
| Too many rows to read | aggregate functions and `GROUP BY` |
| A summary is wanted next to each row | window functions |

Our hash join made a join hundreds of times faster by preparing a dictionary first. The next notebook takes that idea further. What if the database **kept** such a structure permanently, ready for any query that needs it?

---
## ✏️ Exercises

### Exercise 1 (A first join)
List the order id, the date and the customer's **city** for the five most recent orders.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
show("""
    SELECT o.order_id, o.order_date, c.city
    FROM orders AS o
    JOIN customers AS c ON c.customer_id = o.customer_id
    ORDER BY o.order_date DESC, o.order_id DESC
    LIMIT 5
""")
```

*The second sort key makes the result the same on every run. Several orders share the last date, and without a tie-breaker the database may return any five of them.*
</details>

### Exercise 2 (Join by hand)
Using the Python lists `item_rows` (item id, order id, product id, quantity) and `product_rows` (product id, title, category, price), compute the **total revenue** of the shop with a hash join written by hand. Check your answer with one SQL query.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
price_of = {}
for product_id, title, category, price in product_rows:
    price_of[product_id] = price

total = 0.0
for item_id, order_id, product_id, quantity in item_rows:
    total += quantity * price_of[product_id]
print("By hand:", round(total, 2))

print("SQL:    ", round(db.execute("""
    SELECT SUM(i.quantity * p.price) FROM order_items AS i JOIN products AS p ON p.product_id = i.product_id
""").fetchone()[0], 2))
```

*Build the dictionary from the small table and loop over the large one. That is the rule a database follows too: hash the smaller side.*
</details>

### Exercise 3 (Never sold, never bought)
Find (a) the books that have never been sold, and (b) the number of orders of each customer **including** the customers with no orders, showing only the five customers with the fewest.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
show("""
    SELECT p.product_id, p.title
    FROM products AS p
    LEFT JOIN order_items AS i ON i.product_id = p.product_id
    WHERE i.item_id IS NULL
""")

show("""
    SELECT c.name, COUNT(o.order_id) AS orders
    FROM customers AS c
    LEFT JOIN orders AS o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id
    ORDER BY orders, c.customer_id
    LIMIT 5
""")
```

*In (b) the count must be `COUNT(o.order_id)` and not `COUNT(*)`. For a customer with no orders the left join produces one row full of NULLs. `COUNT(*)` would count that row and report one order. `COUNT(column)` ignores NULLs and correctly reports zero.*
</details>

### Exercise 4 (Grouping)
For each **city**, compute the number of customers and their average birth year. Then show only the cities with more than 125 customers, largest first.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
show("""
    SELECT city, COUNT(*) AS customers, ROUND(AVG(birth_year)) AS average_birth_year
    FROM customers
    GROUP BY city
    HAVING COUNT(*) > 125
    ORDER BY customers DESC
""", limit=10)
```

*The condition is on a count, which exists only after grouping, so it belongs in `HAVING`.*
</details>

### Exercise 5 (Find the mistake)
A colleague wants the number of **customers** who bought at least one `Travel` book, and writes the query below. The number is far too high. Explain why, and fix it.

In [ ]:
print(db.execute("""
    SELECT COUNT(*)
    FROM customers AS c
    JOIN orders AS o       ON o.customer_id = c.customer_id
    JOIN order_items AS i  ON i.order_id = o.order_id
    JOIN products AS p     ON p.product_id = i.product_id
    WHERE p.category = 'Travel'
""").fetchone()[0], "customers bought a Travel book?  (there are only", len(customer_rows), "customers)")
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
print(db.execute("""
    SELECT COUNT(DISTINCT c.customer_id)
    FROM customers AS c
    JOIN orders AS o       ON o.customer_id = c.customer_id
    JOIN order_items AS i  ON i.order_id = o.order_id
    JOIN products AS p     ON p.product_id = i.product_id
    WHERE p.category = 'Travel'
""").fetchone()[0], "customers bought at least one Travel book")
```

*After the joins, one row is one order line, so the first query counted the Travel books sold, not the people who bought them. A customer who bought five is counted five times. Counting distinct customer ids gives the intended answer. The result being larger than the number of customers was the warning sign: always compare a result with something you already know.*
</details>

### Exercise 6 (Windows, a little harder)
For each customer of **Chania**, show their orders with the date and a column `order_number` that numbers each customer's orders 1, 2, 3 ... in order of date. Show the first eight rows. (Use `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)`.) Then use it to find how many Chania customers placed at least 15 orders.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
numbered = """
    SELECT c.name, o.order_date,
           ROW_NUMBER() OVER (PARTITION BY c.customer_id ORDER BY o.order_date, o.order_id) AS order_number
    FROM customers AS c
    JOIN orders AS o ON o.customer_id = c.customer_id
    WHERE c.city = 'Chania'
"""
show(numbered + " ORDER BY c.customer_id, order_number", limit=8)

print(db.execute(f"SELECT COUNT(*) FROM ({numbered}) WHERE order_number = 15").fetchone()[0],
      "customers in Chania have placed at least 15 orders")
```

*A customer has a fifteenth order exactly when they have at least fifteen. Numbering rows within a group is also the standard way to pick "the first order of each customer" or "the latest reading of each sensor": number them, then keep number 1.*
</details>

## 🔑 Key takeaways

- A **join** matches rows of two tables on a key. You state the condition, and the database picks the algorithm.
- A **nested loop** join costs the product of the table sizes. A **hash join** builds a dictionary and costs their sum.
- An **inner join** keeps only matching pairs. A **left join** keeps every row of the left table and fills the gaps with NULL.
- **Aggregate functions** with `GROUP BY` give one row per group. `WHERE` filters rows before grouping, `HAVING` filters groups after it.
- A query is evaluated in the order `FROM`, `WHERE`, `GROUP BY`, `HAVING`, `SELECT`, `ORDER BY`, `LIMIT`.
- **Window functions** compute rankings and running totals while keeping every row.
- Joins can **multiply rows**. Always know what one row of your result represents.

---
**Next:** *Notebook 03: Indexes and Query Plans*, where we find out why the same query can take a millisecond or a minute, and how to tell which it will be before running it.

---
*© Ioannis Tsioulis. *From One Table to a Cluster: Working with Data at Scale*. Prepared for educational use.*